# NBA data before Extraction

In [1]:
import pandas as pd


# Load the datasets
file_path = r"C:\Users\Ezra\Downloads\nba_players_details.csv"


df = pd.read_csv(file_path)

# Display the column names and a sample of the data
df_info = df.head()
df_columns = df.columns.tolist()

(df_info, df_columns)

(                                      Player URI        Player Name  \
 0     http://dbpedia.org/resource/Candace_Parker     Candace Parker   
 1  http://dbpedia.org/resource/Quantez_Robertson  Quantez Robertson   
 2     http://dbpedia.org/resource/Scott_Bamforth     Scott Bamforth   
 3       http://dbpedia.org/resource/Becky_Hammon       Becky Hammon   
 4   http://dbpedia.org/resource/Boban_Marjanović   Boban Marjanović   
 
   Has All-Star Selection  Birth Date  Height  \
 0                    Yes  1986-04-19  1.9304   
 1                    Yes  1984-12-16  1.8796   
 2                    Yes  1989-08-12  1.8796   
 3                    Yes  1977-03-11  1.6764   
 4                    Yes  1988-08-15  2.2352   
 
                                             Position  \
 0  http://dbpedia.org/resource/Power_forward_(bas...   
 1         http://dbpedia.org/resource/Shooting_guard   
 2         http://dbpedia.org/resource/Shooting_guard   
 3            http://dbpedia.org/resource/

In [2]:
# Calculate missing values per column
missing_counts = df.isnull().sum()
missing_percentages = (missing_counts / len(df)) * 100

# Combine results into a DataFrame for readability
missing_summary = pd.DataFrame({
    'Missing Values': missing_counts,
    'Percentage (%)': missing_percentages.round(2)
}).sort_values(by='Percentage (%)', ascending=False)

# Print the summary
print("Missing values per column:")
print(missing_summary)

Missing values per column:
                        Missing Values  Percentage (%)
Teams                            11333           62.92
Awards                            5666           31.46
Height                            4024           22.34
Birth Date                         354            1.97
Player URI                           0            0.00
Player Name                          0            0.00
Has All-Star Selection               0            0.00
Position                             0            0.00


Implemtation of a model using the initial dataset

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb

# Step 1: Encode target
df['Has All-Star Selection'] = df['Has All-Star Selection'].map({'Yes': 1, 'No': 0})

# Step 2: Drop non-informative columns
df.drop(['Player URI', 'Player Name', 'Awards'], axis=1, inplace=True)

# Step 3: Simplify URI fields
def extract_label(uri):
    if pd.isna(uri):
        return np.nan
    return uri.split('/')[-1].replace('_', ' ')


# Convert birth date to datetime and extract birth year
df['Birth Date'] = pd.to_datetime(df['Birth Date'], errors='coerce')
df['Birth Year'] = df['Birth Date'].dt.year
df.drop(columns='Birth Date', inplace=True)


# Step 3: Simplify URI fields
def extract_label(uri):
    if pd.isna(uri):
        return np.nan
    return uri.split('/')[-1].replace('_', ' ')

df['Position'] = df['Position'].apply(extract_label)
df['Teams'] = df['Teams'].apply(extract_label)

# Step 4: Handle missing values
df['Height'].fillna(df['Height'].median(), inplace=True)
df['Birth Year'].fillna(df['Birth Year'].median(), inplace=True)
df['Position'].fillna('Unknown', inplace=True)
df['Teams'].fillna('Unknown', inplace=True)

# Step 5: One-hot encode categorical features
df = pd.get_dummies(df, columns=['Position', 'Teams'], drop_first=True)

# Step 6: Split dataset
X = df.drop('Has All-Star Selection', axis=1)
y = df['Has All-Star Selection']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Step 7: Train XGBoost model
model = xgb.XGBClassifier()
model.fit(X_train, y_train)

# Step 8: Evaluate performance
y_pred = model.predict(X_test)
f1 = f1_score(y_test, y_pred)

f1



0.10379241516966067

## NBA Dataset After Extraction

In [4]:
import pandas as pd
import numpy as np

# Load the datasets
file_path = r"C:\Users\Ezra\Downloads\NBA_web_imputation_06_01_2025_10k_samples.csv"

df = pd.read_csv(file_path)

# Display the column names and a sample of the data
df_info = df.head()
df_columns = df.columns.tolist()

(df_info, df_columns)


(                                        Player URI          Player Name  \
 0   http://dbpedia.org/resource/1987_IUSY_Festival   1987 IUSY Festival   
 1      http://dbpedia.org/resource/A'Darius_Pegues      A'Darius Pegues   
 2  http://dbpedia.org/resource/A'Quonesia_Franklin  A'Quonesia Franklin   
 3        http://dbpedia.org/resource/A'dia_Mathies        A'dia Mathies   
 4          http://dbpedia.org/resource/A'ja_Wilson          A'ja Wilson   
 
   Has All-Star Selection  Birth Date  Height  \
 0                     No  1982-01-01     NaN   
 1                     No  1988-03-21  2.0828   
 2                     No  1985-09-29  1.6256   
 3                     No  1991-05-18  1.7526   
 4                    Yes  1996-08-08  1.9558   
 
                                             position  \
 0                                                NaN   
 1    http://dbpedia.org/resource/Center_(basketball)   
 2     http://dbpedia.org/resource/Guard_(basketball)   
 3     http://dbpe

Implementation of a model using the final dataset (after extraction), without prioritization

In [5]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb

# Encode target
df['Has All-Star Selection'] = df['Has All-Star Selection'].map({'Yes': 1, 'No': 0})

# Drop identifiers
df.drop(['Player URI', 'Player Name'], axis=1, inplace=True)

# Convert birth date to datetime and extract birth year
df['Birth Date'] = pd.to_datetime(df['Birth Date'], errors='coerce')
df['Birth Year'] = df['Birth Date'].dt.year
df.drop(columns='Birth Date', inplace=True)

# Simplify and clean 'position' and 'team'
def extract_label(uri):
    if pd.isna(uri):
        return np.nan
    return uri.split('/')[-1].replace('_', ' ')

df['position'] = df['position'].apply(extract_label).fillna('Unknown')
df['team'] = df['team'].apply(extract_label).fillna('Unknown')

# Focus on numeric features and engineer new ones
df['Height'].fillna(df['Height'].median(), inplace=True)
df['height_zscore'] = (df['Height'] - df['Height'].mean()) / df['Height'].std()

# Compute interaction features
df['minutes_x_steals'] = df['minutes per game'] * df['steals per game']
df['fg_x_minutes'] = df['field goal percentage'] * df['minutes per game']

# Crude player value score
df['player_value_score'] = (
    df['field goal percentage'] +
    df['three-point percentage'] +
    df['free throw percentage']
) * df['minutes per game'] + df['defensive rebounds per game'] + df['steals per game']

# Limit position/team cardinality
top_positions = df['position'].value_counts().nlargest(5).index
df['position'] = df['position'].apply(lambda x: x if x in top_positions else 'Other')

top_teams = df['team'].value_counts().nlargest(5).index
df['team'] = df['team'].apply(lambda x: x if x in top_teams else 'Other')


df['height_zscore'] = (df['Height'] - df['Height'].mean()) / df['Height'].std()
df['minutes_x_steals'] = df['minutes per game'] * df['steals per game']
df['fg_x_minutes'] = df['field goal percentage'] * df['minutes per game']
df['player_value_score'] = (
    df['field goal percentage'] +
    df['three-point percentage'] +
    df['free throw percentage']
) * df['minutes per game'] + df['defensive rebounds per game'] + df['steals per game']


# One-hot encode position/team
df = pd.get_dummies(df, columns=['position', 'team'], drop_first=True)

# Fill remaining missing values
df.fillna(0, inplace=True)

# Prepare X and y
X = df.drop('Has All-Star Selection', axis=1)
y = df['Has All-Star Selection']


# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Train XGBoost with improvements
model = xgb.XGBClassifier(
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate F1
y_pred = model.predict(X_test)
f1_enhanced = f1_score(y_test, y_pred)

f1_enhanced


0.3869047619047619

Modeling with Prioritization Methods

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from src.dataset_with_indices_for_full_and_partial_data import Index_Dataset
from src.filtering_methods.filtering_methods_return_index import (RandomSampleFilter, ActiveLearningStartFromCoreSet, FilterEachIterXgboostPathSampleFinal)
import xgboost as xgb

random_state = 42


# ---------- 1. Load & sanitize once ----------
target_col = "Has All-Star Selection"
df_original = df

feature_cols        = [c for c in df_original.columns if c != target_col]
#df_original.columns = (   sanitize_column_names(feature_cols) + [target_col])  # keep label untouched

# ---------- 2. Build X / y ----------
X = df_original.drop(columns=[target_col]).fillna(-999).astype(np.float32)
y = df_original[target_col]

# ---------- 3. Train / test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=random_state
)

# ---------- 4. Feed sanitized structures to Index_Dataset ----------
dataset_partial = Index_Dataset(
    df_original,      # contains the SAME sanitized names
    X_train,
    y_train,
    X_test,
    y_test,
    target_col,
    f1_score,
)

# ---------- 5. Run your filter ----------
all_results   = []
n = len(X_train)
core_set_sizes = [n//10, n//5, n // 3, n // 2, int(0.75 * n), n]

for size in core_set_sizes:
    random_state += 1
    p          = size / n
    model      = xgb.XGBClassifier(eval_metric="logloss", random_state=random_state
                                   )
    sampler    = RandomSampleFilter(name="RandomSampleFilter",
                                    p=p, model=model)
    sampler.test_indices_filter_method(
        dataset_partial, dataset_partial, print_results=True
    )
    all_results.append(
        {"Method": "RandomSampleFilter",
         "Core Set Size": size,
         "F1 Score": sampler.scores,
         "Trial": sampler.run_times,}
    )

    initial_set_size = size //2
    # ExtendTAB Filter(Active Learning Start From Core Set)
    core_tab = FilterEachIterXgboostPathSampleFinal(
        'test_each_iter',
        trees_to_stop=30,
        threshold=5,
        examples_to_keep=size//2,
        prediction_model=model
    )
    core_set_sampler = core_tab.sample_indices(
        dataset_partial,
        p = initial_set_size / len(dataset_partial.X_train)
    )

    sampler = ActiveLearningStartFromCoreSet(name="ExtenTab",number_of_examples=size, start_size=size//2,core_set_sampler=core_set_sampler, batch_size=100,
                                     model=model)
    sampler.test_indices_filter_method(
        dataset_partial, dataset_partial, print_results=True
    )
    all_results.append(
        {"Method": "ExtenTab",
         "Core Set Size": size,
         "F1 Score": sampler.scores,
         "Trial": sampler.run_times}
    )


0 0.09998648831239022
Random Sampler: 0 727
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.18528610354223432, filtered_score=0.18528610354223432, run_time=0, new_size_percent=0.0982299689231185)
ExtendTab: len(initial_indices) 370 start_size 370
start size 370
ExtendTab: AL Phase Current training set size: 370
ExtendTab: AL Phase Current training set size: 470
ExtendTab: AL Phase Current training set size: 570
ExtendTab: AL Phase Current training set size: 670
validate that the training set size is the wished size training set size: 770 wished core-set size: 740
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.3548387096774193, filtered_score=0.3548387096774193, run_time=0, new_size_percent=0.10403999459532495)
0 0.19997297662478045
Random Sampler: 0 1408
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.238888888

In [21]:


# Step 1: Flatten F1 scores into rows
flattened = []
for entry in all_results:
    method = entry['Method']
    core_size = entry['Core Set Size']
    for score in entry['F1 Score']:
        flattened.append({'Method': method, 'Core Set Size': core_size, 'F1 Score': score})

df = pd.DataFrame(flattened)
# Step 2: Group and summarize
summary = df.groupby(['Method', 'Core Set Size']).agg(
    avg_f1_score=('F1 Score', 'mean'),
    #std_f1_score=('F1 Score', 'std'),
    num_trials=('F1 Score', 'count')
).reset_index()

# Optional: Mark best method per core set size
summary['is_best'] = summary.groupby('Core Set Size')['avg_f1_score'].transform(
    lambda x: x == x.max()
)

# Step 3: Print or export
(summary.sort_values(['Method', 'Core Set Size', 'avg_f1_score'], ascending=[True, True,False]))

,Method,Core Set Size,avg_f1_score,std_f1_score,num_trials,is_best
0,ExtenTab,740,0.354839,NaN,1,True
1,ExtenTab,1480,0.393782,NaN,1,True
2,ExtenTab,2467,0.372973,NaN,1,True
3,ExtenTab,3700,0.359756,NaN,1,True
4,ExtenTab,5550,0.366972,NaN,1,True
5,ExtenTab,7401,0.391691,NaN,1,True
6,RandomSampleFilter,740,0.185286,NaN,1,False
7,RandomSampleFilter,1480,0.238889,NaN,1,False
8,RandomSampleFilter,2467,0.252055,NaN,1,False
9,RandomSampleFilter,3700,0.340782,NaN,1,False


In [11]:

# Normalize the results so each trial gets its own row
expanded_results = []
for entry in all_results:
    scores = entry["F1 Score"]
    trials = entry["Trial"]
    method = entry["Method"]
    core_size = entry["Core Set Size"]

    for trial_idx, score in enumerate(scores):
        expanded_results.append({
            "Method": method,
            "Core Set Size": core_size,
            "F1 Score": score
        })

# Convert to DataFrame
results_df = pd.DataFrame(expanded_results)

# Save to CSV
results_df.to_csv("animal_endangerment_core_set_results.csv", index=False)

# (Optional) Display it
print(results_df)


                Method  Core Set Size  F1 Score
0   RandomSampleFilter            740  0.185286
1             ExtenTab            740  0.354839
2   RandomSampleFilter           1480  0.238889
3             ExtenTab           1480  0.393782
4   RandomSampleFilter           2467  0.252055
5             ExtenTab           2467  0.372973
6   RandomSampleFilter           3700  0.340782
7             ExtenTab           3700  0.359756
8   RandomSampleFilter           5550  0.319277
9             ExtenTab           5550  0.366972
10  RandomSampleFilter           7401  0.343373
11            ExtenTab           7401  0.391691
12  Extentab Free size           7401  0.352584


## ExtendTab Free Size

In [9]:
from src.filtering_methods.filtering_methods_return_index import (ActiveLearningStartFromCoreSet_based_on_reccommended_sampling_sizes)

# ===============================
#   ExtendTAB (Free size)
# ===============================
core_tab_hard_Samples = FilterEachIterXgboostPathSampleFinal(
    'test_each_iter',
    trees_to_stop=30,
    threshold=5,
    prediction_model=model,
    examples_to_keep=len(dataset_partial.X_train)//2,
)
core_set_sampler = core_tab_hard_Samples.sample_indices(
    dataset_partial,
    p=initial_set_size / len(dataset_partial.X_train)
)

filter_instance = ActiveLearningStartFromCoreSet_based_on_reccommended_sampling_sizes(
    name='ExtendTAB',
    start_size=len(dataset_partial.X_train)//2,
    number_of_examples=len(dataset_partial.X_train),
    model=model,
    core_set_sampler=core_set_sampler,
    batch_size=100
)

# Again, the critical fix: call the method
filter_instance.test_indices_filter_method(dataset_partial, dataset_partial, print_results=True)

all_results.append({
    #'Dataset': dataset_name,
    'Method': 'Extentab Free size',
    'Core Set Size': filter_instance.number_of_examples,
    'F1 Score': filter_instance.scores,
     'Data Used (%)': filter_instance.new_size_percents * 100
})


[ExtendTab] using supplied core set of 3700/3700
[ExtendTab] AL-phase — train size 3700
[ExtendTab] val-score 0.3784
[ExtendTab] AL-phase — train size 3800
[ExtendTab] val-score 0.3818
[ExtendTab] AL-phase — train size 3900
[ExtendTab] val-score 0.3684
[ExtendTab] AL-phase — train size 4000
[ExtendTab] val-score 0.3609
[ExtendTab] AL-phase — train size 4100
[ExtendTab] val-score 0.3537
[ExtendTab] AL-phase — train size 4200
[ExtendTab] val-score 0.3354
[ExtendTab] Stopped after 5 flat rounds.
[ExtendTab] initial 3700 → final 4300 (time 2.0s)
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.3525835866261398, filtered_score=0.3525835866261398, run_time=0, new_size_percent=0.5810025672206458)
